In [0]:
%sql
CREATE CATALOG IF NOT EXISTS ATMOS
  MANAGED LOCATION 'abfss://atmos-landing-dev-001@saatmosdevwestus2001.dfs.core.windows.net/';

CREATE SCHEMA   IF NOT EXISTS ATMOS.SILVER;

In [0]:
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("ingestion_date", "", "Data de processamento (YYYY-MM-DD)")
ingestion_date = dbutils.widgets.get("ingestion_date")

In [0]:
df_inmet = (
    spark.table("atmos.silver.climate_inmet")
    .filter(F.col("data_ingestao") == ingestion_date)
)

df_visual_crossing = (
    spark.table("atmos.silver.climate_visual_crossing")
    .filter(F.col("data_ingestao") == ingestion_date)
)

In [0]:
df_unified = df_inmet.unionByName(df_visual_crossing)

In [0]:
(
    df_unified.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"data_ingestao = '{ingestion_date}'")
    .partitionBy("data_ingestao")
    .saveAsTable("atmos.silver.climate_unified")
)

In [0]:
count = (
    spark.table("atmos.silver.climate_unified")
    .filter(F.col("data_ingestao") == ingestion_date)
    .count()
)

assert count > 0, f"Nenhum registro gravado em silver.climate_unified para data_ingestao={ingestion_date}."
print(f"OK — {count} registros em silver.climate_unified | data_ingestao={ingestion_date}")